# Flight Delay Prediction (Full-Stack ML)

### Variables List

**Time/Date**

* FL_DATE — flight date

**Carrier**

* OP_UNIQUE_CARRIER — carrier code

**Route**

* ORIGIN — origin airport
* DEST — destination airport
* DISTANCE — non-stop distance

**Departure performance**

* CRS_DEP_TIME — scheduled departure time
* DEP_TIME — actual departure time
* DEP_DELAY — departure delay (minutes)

**Arrival performance**

* CRS_ARR_TIME — scheduled arrival time
* ARR_TIME — actual arrival time
* ARR_DELAY — arrival delay (minutes) -> (main classification/regression target)

**Cancellations/diversions**

* CANCELLED
* DIVERTED

**Cause of delay (post-hoc, exclude from model features, leakage)**

* CARRIER_DELAY
* WEATHER_DELAY
* NAS_DELAY (National Air System Delay)
* SECURITY_DELAY
* LATE_AIRCRAFT_DELAY

* Combining all 6 csv's

In [1]:
import pandas as pd
import glob
import os
from constants import *

flight_data_dir = os.path.join(DATASETS_DIR, "flight_data")
csv_files = glob.glob(os.path.join(flight_data_dir, "*.csv"))
print(f"Found {len(csv_files)} files:", csv_files)

# read & combine
dfs = []
for file in csv_files:
    df = pd.read_csv(file, low_memory=False)
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

# Drop the stray trailing "Unnamed" column BTS exports often add
combined = combined.loc[:, ~combined.columns.str.contains('^Unnamed')]

print(f"Combined shape: {combined.shape}")
print(f"Date range: {combined['FL_DATE'].min()} to {combined['FL_DATE'].max()}")
print(combined.columns.tolist())

# saving merged dataset
combined.to_csv(os.path.join(DATASETS_DIR, "flight_data_combined.csv"), index=False)

Found 6 files: ['D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\apr_flight_data.csv', 'D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\feb_flight_data.csv', 'D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\jan_flight_data.csv', 'D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\june_flight_data.csv', 'D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\mar_flight_data.csv', 'D:/CitrusBits/pythonic-rebirth\\datasets\\flight_data\\may_flight_data.csv']
Combined shape: (3446676, 18)
Date range: 1/1/2025 12:00:00 AM to 6/9/2025 12:00:00 AM
['FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']


In [2]:
# Parse FL_DATE properly first so month grouping works correctly
combined['FL_DATE'] = pd.to_datetime(combined['FL_DATE'])

print("Shape:", combined.shape)
print("\nDate range:", combined['FL_DATE'].min(), "to", combined['FL_DATE'].max())
print("\nRows per month:")
print(combined['FL_DATE'].dt.to_period('M').value_counts().sort_index())

print("\nMissing values (top offenders):")
print(combined.isnull().sum().sort_values(ascending=False).head(15))

print("\nCancelled flights:", combined['CANCELLED'].sum())
print("Diverted flights:", combined['DIVERTED'].sum())

print("\nUnique carriers:",
      combined['OP_UNIQUE_CARRIER'].nunique() if 'OP_UNIQUE_CARRIER' in combined.columns else "column not found")
print("Unique origin airports:", combined['ORIGIN'].nunique())
print("Unique destination airports:", combined['DEST'].nunique())

C:\Users\Hp Pavilion 13 Aero\AppData\Local\Temp\ipykernel_13128\232528155.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined['FL_DATE'] = pd.to_datetime(combined['FL_DATE'])


Shape: (3446676, 18)

Date range: 2025-01-01 00:00:00 to 2025-06-30 00:00:00

Rows per month:
FL_DATE
2025-01    539747
2025-02    504884
2025-03    600872
2025-04    583950
2025-05    605648
2025-06    611575
Freq: M, Name: count, dtype: int64

Missing values (top offenders):
NAS_DELAY              2705363
WEATHER_DELAY          2705363
CARRIER_DELAY          2705363
LATE_AIRCRAFT_DELAY    2705363
SECURITY_DELAY         2705363
ARR_DELAY                60854
ARR_TIME                 52610
DEP_DELAY                49527
DEP_TIME                 49335
FL_DATE                      0
OP_UNIQUE_CARRIER            0
ORIGIN                       0
DEST                         0
CRS_DEP_TIME                 0
CRS_ARR_TIME                 0
dtype: int64

Cancelled flights: 51634.0
Diverted flights: 9220.0

Unique carriers: 14
Unique origin airports: 346
Unique destination airports: 346


Missing ARR_DELAY does NOT mean the flight was on time. If a flight arrived on time or early, BTS still records an actual ARR_DELAY value, it'd just be 0 or negative (e.g., -8 for 8 minutes early). A missing value in ARR_DELAY specifically means no arrival data exists at all, because the flight was either:

Cancelled means never flew, so there's nothing to measure
Diverted means landed somewhere other than the scheduled destination, so "arrival delay at DEST" doesn't cleanly apply

In [3]:
# first we will work with ARR_DELAY variable
missing_arr = combined[combined['ARR_DELAY'].isnull()]
print("Missing ARR_DELAY rows:", len(missing_arr))
print("Of those, cancelled:", missing_arr['CANCELLED'].sum())
print("Of those, diverted:", missing_arr['DIVERTED'].sum())

Missing ARR_DELAY rows: 60854
Of those, cancelled: 51634.0
Of those, diverted: 9220.0


In [4]:
print("Before drop:", combined.shape)

# Keep only flights that actually completed (not canceled, not diverted)
combined = combined[(combined['CANCELLED'] == 0) & (combined['DIVERTED'] == 0)].copy()

print("After drop:", combined.shape)

# checking, should now be 0 missing in these columns
print()
print("Remaining missing values:")
print(combined[['ARR_DELAY', 'DEP_DELAY', 'ARR_TIME', 'DEP_TIME']].isnull().sum())

Before drop: (3446676, 18)
After drop: (3385822, 18)

Remaining missing values:
ARR_DELAY    0
DEP_DELAY    0
ARR_TIME     0
DEP_TIME     0
dtype: int64


In [5]:
print("Remaining missing values across all columns:")
print(combined.isnull().sum().sort_values(ascending=False))

print("\nTotal rows:", len(combined))

Remaining missing values across all columns:
NAS_DELAY              2644509
WEATHER_DELAY          2644509
CARRIER_DELAY          2644509
LATE_AIRCRAFT_DELAY    2644509
SECURITY_DELAY         2644509
FL_DATE                      0
OP_UNIQUE_CARRIER            0
ORIGIN                       0
CRS_DEP_TIME                 0
DEST                         0
ARR_DELAY                    0
ARR_TIME                     0
CRS_ARR_TIME                 0
DEP_DELAY                    0
DEP_TIME                     0
DISTANCE                     0
CANCELLED                    0
DIVERTED                     0
dtype: int64

Total rows: 3385822


It is in military timing
In CRS_DEP_TIME 500 means 5:00 AM , 1308 means 1:08 PM

In [6]:
# Convert scheduled/actual times from HHMM int format into usable hour buckets
# converting the HHMM time columns.
time_cols = ['CRS_DEP_TIME', 'DEP_TIME', 'CRS_ARR_TIME', 'ARR_TIME']

for col in time_cols:
    # HHMM format edge case: BTS sometimes uses 2400 for midnight — normalize to 0
    combined[col] = combined[col].replace(2400, 0)
    combined[f'{col}_HOUR'] = (combined[col] // 100).astype(int)

for col in ['DEP_TIME', 'ARR_TIME']:
    combined[col] = combined[col].astype(int)

# Quick check
print(combined[[c for col in time_cols for c in (col, f'{col}_HOUR')]].head())
print("\nHour range check (should be 0-23):")
for col in time_cols:
    print(f"{col}_HOUR:", combined[f'{col}_HOUR'].min(), "to", combined[f'{col}_HOUR'].max())

   CRS_DEP_TIME  CRS_DEP_TIME_HOUR  DEP_TIME  DEP_TIME_HOUR  CRS_ARR_TIME  \
0           500                  5       609              6           750   
1           600                  6       552              5           849   
2           820                  8       818              8          1110   
3           955                  9      1045             10          1244   
4          1308                 13      1334             13          1555   

   CRS_ARR_TIME_HOUR  ARR_TIME  ARR_TIME_HOUR  
0                  7       858              8  
1                  8       838              8  
2                 11      1059             10  
3                 12      1316             13  
4                 15      1605             16  

Hour range check (should be 0-23):
CRS_DEP_TIME_HOUR: 0 to 23
DEP_TIME_HOUR: 0 to 23
CRS_ARR_TIME_HOUR: 0 to 23
ARR_TIME_HOUR: 0 to 23


In [7]:
# building the Regression target variables.
print("ARR_DELAY stats:")
print(combined['ARR_DELAY'].describe())


# Classification target — bucket into Early / On-time / Delayed
def classify_delay(delay):
    if delay <= -15:
        return 'Early'
    elif delay <= 15:
        return 'On-time'
    else:
        return 'Delayed'


combined['DELAY_CLASS'] = combined['ARR_DELAY'].apply(classify_delay)

print("\nClass distribution:")
print(combined['DELAY_CLASS'].value_counts())
print(combined['DELAY_CLASS'].value_counts(normalize=True))

ARR_DELAY stats:
count    3.385822e+06
mean     7.897115e+00
std      5.893396e+01
min     -1.280000e+02
25%     -1.600000e+01
50%     -6.000000e+00
75%      1.100000e+01
max      3.407000e+03
Name: ARR_DELAY, dtype: float64

Class distribution:
DELAY_CLASS
On-time    1717082
Early       950317
Delayed     718423
Name: count, dtype: int64
DELAY_CLASS
On-time    0.507139
Early      0.280675
Delayed    0.212186
Name: proportion, dtype: float64


In [8]:
# Check how extreme the outliers are
print("Flights with ARR_DELAY > 300 min (5+ hrs):", (combined['ARR_DELAY'] > 300).sum())
print("Flights with ARR_DELAY > 600 min (10+ hrs):", (combined['ARR_DELAY'] > 600).sum())
print("Flights with ARR_DELAY > 1000 min:", (combined['ARR_DELAY'] > 1000).sum())

combined[combined['ARR_DELAY'] > 1000][
    ['FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY', 'ARR_DELAY']].head(10)

Flights with ARR_DELAY > 300 min (5+ hrs): 17061
Flights with ARR_DELAY > 600 min (10+ hrs): 4794
Flights with ARR_DELAY > 1000 min: 1474


,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,DEP_DELAY,ARR_DELAY
182,2025-04-01,AA,CLE,DFW,1863.0,1886.0
1855,2025-04-01,AA,MSY,PHX,2186.0,2177.0
2021,2025-04-01,AA,ORF,DFW,1703.0,1779.0
5174,2025-04-01,DL,GEG,MSP,1032.0,1034.0
20706,2025-04-02,AA,MSY,DFW,1326.0,1313.0
20845,2025-04-02,AA,ORD,PHL,1167.0,1163.0
22655,2025-04-02,B6,ORD,BOS,1240.0,1220.0
26583,2025-04-02,MQ,CAE,DFW,2074.0,2032.0
27995,2025-04-02,OH,BNA,CLT,1313.0,1306.0
28016,2025-04-02,OH,CAE,CLT,1005.0,1020.0


**Capping**
Basically we are using this concept to cap the upper bound range for flights with arr_delay since both of these delays are due to extreme unlikely conditions which have very less chance of happening again in future


> The reasoning: These 17K rows aren't noise or errors. They're mostly real, severely delayed flights.

> The actual problem isn't that they exist, it's the scale of the raw numbers. A regression model trained with typical loss functions (like MSE) squares the error. If one flight has an actual delay of 3407 minutes and your model predicts even 1000 minutes for it, that's an error of ~2400 minutes and when squared, that's a massive number that can dwarf the combined error from thousands of normal flights. The model ends up distorting itself trying to chase these few extreme values, at the cost of accuracy on the vast majority of realistic cases (5–60 minute delays, which is what 99%+ of your users will actually care about).

> Note
* ARR_DELAY → keep using this for your DELAY_CLASS classification target and for EDA
* ARR_DELAY_CAPPED → this is what you'll train your regression model on later

In [9]:
# Cap ARR_DELAY at 300 minutes for the regression target
combined['ARR_DELAY_CAPPED'] = combined['ARR_DELAY'].clip(upper=300)

print("Before capping - max:", combined['ARR_DELAY'].max())
print("After capping - max:", combined['ARR_DELAY_CAPPED'].max())

print("\nHow many rows were actually affected:")
print((combined['ARR_DELAY'] != combined['ARR_DELAY_CAPPED']).sum())

print("\nNew stats:")
print(combined['ARR_DELAY_CAPPED'].describe())

Before capping - max: 3407.0
After capping - max: 300.0

How many rows were actually affected:
17061

New stats:
count    3.385822e+06
mean     6.667413e+00
std      4.485950e+01
min     -1.280000e+02
25%     -1.600000e+01
50%     -6.000000e+00
75%      1.100000e+01
max      3.000000e+02
Name: ARR_DELAY_CAPPED, dtype: float64


In [10]:
import holidays

combined['MONTH'] = combined['FL_DATE'].dt.month
combined['DAY_OF_WEEK'] = combined['FL_DATE'].dt.dayofweek  # 0=Monday, 6=Sunday
combined['DAY_OF_MONTH'] = combined['FL_DATE'].dt.day
combined['IS_WEEKEND'] = combined['DAY_OF_WEEK'].isin([5, 6]).astype(int)

# US holidays — flag if the flight date is a federal holiday
us_holidays = holidays.US(years=[2025, 2026])
combined['IS_HOLIDAY'] = combined['FL_DATE'].isin(us_holidays).astype(int)

print(combined[['FL_DATE', 'MONTH', 'DAY_OF_WEEK', 'DAY_OF_MONTH', 'IS_WEEKEND', 'IS_HOLIDAY']].head(10))
print("\nWeekend flight count:", combined['IS_WEEKEND'].sum())
print("Holiday flight count:", combined['IS_HOLIDAY'].sum())
print("\nDelay rate by day of week:")
print(combined.groupby('DAY_OF_WEEK')['ARR_DELAY'].mean())

     FL_DATE  MONTH  DAY_OF_WEEK  DAY_OF_MONTH  IS_WEEKEND  IS_HOLIDAY
0 2025-04-01      4            1             1           0           0
1 2025-04-01      4            1             1           0           0
2 2025-04-01      4            1             1           0           0
3 2025-04-01      4            1             1           0           0
4 2025-04-01      4            1             1           0           0
5 2025-04-01      4            1             1           0           0
6 2025-04-01      4            1             1           0           0
7 2025-04-01      4            1             1           0           0
8 2025-04-01      4            1             1           0           0
9 2025-04-01      4            1             1           0           0

Weekend flight count: 948962
Holiday flight count: 0

Delay rate by day of week:
DAY_OF_WEEK
0     9.537789
1     3.110872
2     5.172721
3     9.370296
4     9.337734
5     5.709533
6    11.900922
Name: ARR_DELAY, dty

In [11]:
us_holidays = holidays.US(years=[2025, 2026])
combined['IS_HOLIDAY'] = combined['FL_DATE'].dt.date.isin(us_holidays).astype(int)

print("Holiday flight count:", combined['IS_HOLIDAY'].sum())

Holiday flight count: 94967


> Weekend delays are not only more but also are more unpredictable

In [12]:
print(combined.groupby('IS_HOLIDAY')['ARR_DELAY'].mean())
print(combined.groupby('IS_HOLIDAY')['ARR_DELAY'].describe())

IS_HOLIDAY
0     7.633011
1    17.048985
Name: ARR_DELAY, dtype: float64
                count       mean        std    min   25%  50%   75%     max
IS_HOLIDAY                                                                 
0           3290855.0   7.633011  58.542457 -128.0 -16.0 -6.0  10.0  3407.0
1             94967.0  17.048985  70.575024  -68.0 -13.0 -2.0  21.0  2628.0


In [13]:
for col in ['ORIGIN', 'DEST', 'OP_UNIQUE_CARRIER']:
    freq = combined[col].value_counts(normalize=True)
    combined[f'{col}_FREQ'] = combined[col].map(freq)

# Quick check
print(combined[['ORIGIN', 'ORIGIN_FREQ', 'DEST', 'DEST_FREQ', 'OP_UNIQUE_CARRIER', 'OP_UNIQUE_CARRIER_FREQ']].head(10))

print("\nTop 5 busiest origin airports:")
print(combined['ORIGIN'].value_counts().head())

print("\nCarrier flight volume:")
print(combined['OP_UNIQUE_CARRIER'].value_counts())

  ORIGIN  ORIGIN_FREQ DEST  DEST_FREQ OP_UNIQUE_CARRIER  \
0    ABQ     0.003557  DFW   0.044318                AA   
1    ABQ     0.003557  DFW   0.044318                AA   
2    ABQ     0.003557  DFW   0.044318                AA   
3    ABQ     0.003557  DFW   0.044318                AA   
4    ABQ     0.003557  DFW   0.044318                AA   
5    ABQ     0.003557  DFW   0.044318                AA   
6    ABQ     0.003557  PHX   0.029799                AA   
7    ABQ     0.003557  PHX   0.029799                AA   
8    ABQ     0.003557  PHX   0.029799                AA   
9    ALB     0.001739  CLT   0.029346                AA   

   OP_UNIQUE_CARRIER_FREQ  
0                0.138894  
1                0.138894  
2                0.138894  
3                0.138894  
4                0.138894  
5                0.138894  
6                0.138894  
7                0.138894  
8                0.138894  
9                0.138894  

Top 5 busiest origin airports:
ORIGIN
DEN

In [14]:
print("DISTANCE stats:")
print(combined['DISTANCE'].describe())

# Sanity checks
print("\nAny zero or negative distances?", (combined['DISTANCE'] <= 0).sum())
print("Any missing distances?", combined['DISTANCE'].isnull().sum())

# Shortest and longest routes — do they make real-world sense?
print("\nShortest routes:")
print(combined.nsmallest(5, 'DISTANCE')[['ORIGIN', 'DEST', 'DISTANCE']])

print("\nLongest routes:")
print(combined.nlargest(5, 'DISTANCE')[['ORIGIN', 'DEST', 'DISTANCE']])

# Check same-airport-as-origin-and-dest — data error, not a real flight
print("\nRows where ORIGIN == DEST:", (combined['ORIGIN'] == combined['DEST']).sum())

DISTANCE stats:
count    3.385822e+06
mean     8.485462e+02
std      6.018336e+02
min      3.100000e+01
25%      4.040000e+02
50%      7.010000e+02
75%      1.086000e+03
max      5.095000e+03
Name: DISTANCE, dtype: float64

Any zero or negative distances? 0
Any missing distances? 0

Shortest routes:
      ORIGIN DEST  DISTANCE
3026     PSG  WRG      31.0
3316     WRG  PSG      31.0
21886    PSG  WRG      31.0
22174    WRG  PSG      31.0
40982    PSG  WRG      31.0

Longest routes:
      ORIGIN DEST  DISTANCE
4690     BOS  HNL    5095.0
5209     HNL  BOS    5095.0
7438     BOS  HNL    5095.0
23544    BOS  HNL    5095.0
24056    HNL  BOS    5095.0

Rows where ORIGIN == DEST: 0


This is a real, meaningful pattern, not a bug — early summer (May-June) genuinely does see more delays than winter/spring months, driven by things like summer thunderstorm season ramping up, higher passenger volume as summer travel begins, and airport congestion increasing. This is a much more honest test than a random split would give you (a random split would blend Jan and June together, hiding this seasonal effect entirely and making your reported accuracy look artificially rosy). Here we did Jan to Apr for training and May to June for testing.


In [15]:
import airportsdata

airports = airportsdata.load('IATA')  # dict keyed by IATA code

# Build a lookup DataFrame for the airports actually in your dataset
unique_airports = pd.unique(combined[['ORIGIN', 'DEST']].values.ravel())
print(f"Unique airports in dataset: {len(unique_airports)}")

airport_coords = []
missing = []
for code in unique_airports:
    if code in airports:
        airport_coords.append({
            'AIRPORT': code,
            'LAT': airports[code]['lat'],
            'LON': airports[code]['lon']
        })
    else:
        missing.append(code)

airport_coords_df = pd.DataFrame(airport_coords)
print(f"Matched: {len(airport_coords_df)}")
print(f"Missing/unmatched codes: {missing}")
print(airport_coords_df.head())

Unique airports in dataset: 346
Matched: 346
Missing/unmatched codes: []
  AIRPORT        LAT         LON
0     ABQ  35.038932 -106.608262
1     DFW  32.897233  -97.037695
2     PHX  33.434278 -112.011583
3     ALB  42.749116  -73.801980
4     CLT  35.213187  -80.951379


In [16]:
import meteostat as ms
from datetime import datetime
import pandas as pd

start = datetime(2025, 1, 1)
end = datetime(2025, 6, 30, 23, 59)

weather_data = {}
failed = []
airport_to_station = {}

for idx, row in airport_coords_df.iterrows():
    code = row['AIRPORT']
    try:
        point = ms.Point(row['LAT'], row['LON'])
        nearby = ms.stations.nearby(point, limit=1)

        if nearby.empty:
            failed.append(code)
            continue

        station_id = nearby.index[0]
        airport_to_station[code] = station_id

        ts = ms.hourly(ms.Station(id=station_id), start, end)
        data = ts.fetch()

        if data is not None and not data.empty:
            weather_data[code] = data
        else:
            failed.append(code)

    except Exception as e:
        failed.append(code)

    if idx % 25 == 0:
        print(f"Processed {idx}/{len(airport_coords_df)} airports... (success so far: {len(weather_data)})")

print(f"\nFinal: Successfully pulled weather for {len(weather_data)} airports")
print(f"Failed/empty: {len(failed)}")
print(failed)

Processed 0/346 airports... (success so far: 1)
Processed 25/346 airports... (success so far: 26)
Processed 50/346 airports... (success so far: 51)
Processed 75/346 airports... (success so far: 76)
Processed 100/346 airports... (success so far: 100)
Processed 125/346 airports... (success so far: 125)
Processed 150/346 airports... (success so far: 150)
Processed 175/346 airports... (success so far: 175)
Processed 200/346 airports... (success so far: 200)
Processed 225/346 airports... (success so far: 224)
Processed 250/346 airports... (success so far: 249)
Processed 275/346 airports... (success so far: 273)
Processed 300/346 airports... (success so far: 297)
Processed 325/346 airports... (success so far: 322)

Final: Successfully pulled weather for 342 airports
Failed/empty: 4
['HNL', 'FSM', 'DDC', 'SGU']


In [17]:
# Combine all per-airport weather DataFrames into one long DataFrame with an AIRPORT column
weather_frames = []
for code, df in weather_data.items():
    df = df.copy()
    df['AIRPORT'] = code
    df = df.reset_index()  # 'time' becomes a column instead of index
    weather_frames.append(df)

all_weather = pd.concat(weather_frames, ignore_index=True)
print(all_weather.shape)
print(all_weather.head())
print(all_weather['AIRPORT'].nunique(), "unique airports in weather data")

(1480878, 13)
                 time  temp  rhum  prcp  snwd  wdir  wspd  wpgt    pres  tsun  \
0 2025-01-01 00:00:00  10.6    26   0.0  <NA>    10   5.4  <NA>  1015.4  <NA>   
1 2025-01-01 01:00:00   8.9    29   0.0  <NA>   340   5.4  <NA>  1016.9  <NA>   
2 2025-01-01 02:00:00   8.9    30   0.0  <NA>   350  18.4  <NA>  1018.1  <NA>   
3 2025-01-01 03:00:00   7.2    31   0.0  <NA>   340  16.6  <NA>  1019.0  <NA>   
4 2025-01-01 04:00:00   6.1    32   0.0  <NA>   350  13.0  <NA>  1019.8  <NA>   

   cldc  coco AIRPORT  
0     3     2     ABQ  
1     0     2     ABQ  
2     0     2     ABQ  
3     0     1     ABQ  
4     0     1     ABQ  
342 unique airports in weather data


In [18]:
# now actual merging
# Build a merge key on the flight side: combine FL_DATE + CRS_DEP_TIME_HOUR into one datetime
combined['WEATHER_MERGE_TIME'] = combined['FL_DATE'] + pd.to_timedelta(combined['CRS_DEP_TIME_HOUR'], unit='h')

# Build the matching merge key on the weather side (already a clean datetime, just rename for clarity)
all_weather['WEATHER_MERGE_TIME'] = all_weather['time']

# Merge: match on ORIGIN airport + the combined date-hour
merged = combined.merge(
    all_weather,
    left_on=['ORIGIN', 'WEATHER_MERGE_TIME'],
    right_on=['AIRPORT', 'WEATHER_MERGE_TIME'],
    how='left'
)

print("Before merge:", combined.shape)
print("After merge:", merged.shape)

# Check how many rows successfully got weather data
weather_cols = ['temp', 'rhum', 'prcp', 'wspd', 'pres', 'coco']
print("\nMissing weather values after merge:")
print(merged[weather_cols].isnull().sum())
print(f"\n% rows with missing weather: {merged['temp'].isnull().mean() * 100:.2f}%")

Before merge: (3385822, 33)
After merge: (3385822, 46)

Missing weather values after merge:
temp     35600
rhum     35649
prcp    235530
wspd     35673
pres     38756
coco     71530
dtype: int64

% rows with missing weather: 1.05%


In [19]:
# Fill all weather columns before splitting
merged['prcp'] = merged['prcp'].fillna(0)

merged['coco'] = merged['coco'].astype('Int16')
merged['coco'] = merged['coco'].fillna(-1)

weather_fill_cols = ['temp', 'rhum', 'wspd', 'pres']
merged = merged.sort_values(['ORIGIN', 'WEATHER_MERGE_TIME'])
merged[weather_fill_cols] = merged.groupby('ORIGIN')[weather_fill_cols].transform(lambda x: x.ffill().bfill())

# Final check across all weather columns
print(merged[['temp', 'rhum', 'prcp', 'wspd', 'pres', 'coco']].isnull().sum())

temp    32663
rhum    32663
prcp        0
wspd    32663
pres    32663
coco        0
dtype: int64


In [20]:
weather_fill_cols = ['temp', 'rhum', 'wspd', 'pres']

# Cast to nullable Float64 first so decimal fill values are allowed
for col in weather_fill_cols:
    merged[col] = merged[col].astype('Float64')

for col in weather_fill_cols:
    merged[col] = merged[col].fillna(merged[col].mean())

print("After safety fill:")
print(merged[weather_fill_cols].isnull().sum())

After safety fill:
temp    0
rhum    0
wspd    0
pres    0
dtype: int64


In [21]:
# now saving this merged dataset
import os

output_path = os.path.join(DATASETS_DIR, "flight_weather_merged.csv")
merged.to_csv(output_path, index=False)
print(f"Saved: {merged.shape} to {output_path}")

Saved: (3385822, 46) to D:/CitrusBits/pythonic-rebirth\datasets\flight_weather_merged.csv


In [22]:
# Time-based split — train on Jan-Apr, test on May-June
split_date = pd.Timestamp('2025-05-01')

train_df = merged[merged['FL_DATE'] < split_date].copy()
test_df = merged[merged['FL_DATE'] >= split_date].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain date range:", train_df['FL_DATE'].min(), "to", train_df['FL_DATE'].max())
print("Test date range:", test_df['FL_DATE'].min(), "to", test_df['FL_DATE'].max())

print("\nTrain class balance:")
print(train_df['DELAY_CLASS'].value_counts(normalize=True))
print("\nTest class balance:")
print(test_df['DELAY_CLASS'].value_counts(normalize=True))

Train shape: (2188776, 46)
Test shape: (1197046, 46)

Train date range: 2025-01-01 00:00:00 to 2025-04-30 00:00:00
Test date range: 2025-05-01 00:00:00 to 2025-06-30 00:00:00

Train class balance:
DELAY_CLASS
On-time    0.500000
Early      0.309531
Delayed    0.190470
Name: proportion, dtype: float64

Test class balance:
DELAY_CLASS
On-time    0.520193
Delayed    0.251893
Early      0.227914
Name: proportion, dtype: float64


In [23]:
# Compute means using ONLY the training set
origin_means = train_df.groupby('ORIGIN')['ARR_DELAY'].mean()
dest_means = train_df.groupby('DEST')['ARR_DELAY'].mean()
carrier_means = train_df.groupby('OP_UNIQUE_CARRIER')['ARR_DELAY'].mean()

# Apply to both train and test (test uses train's means — this is what avoids leakage)
train_df['ORIGIN_DELAY_MEAN'] = train_df['ORIGIN'].map(origin_means)
test_df['ORIGIN_DELAY_MEAN'] = test_df['ORIGIN'].map(origin_means)

train_df['DEST_DELAY_MEAN'] = train_df['DEST'].map(dest_means)
test_df['DEST_DELAY_MEAN'] = test_df['DEST'].map(dest_means)

train_df['CARRIER_DELAY_MEAN'] = train_df['OP_UNIQUE_CARRIER'].map(carrier_means)
test_df['CARRIER_DELAY_MEAN'] = test_df['OP_UNIQUE_CARRIER'].map(carrier_means)

# Check for any NaNs in test (would happen if test has an airport/carrier never seen in train)
print("NaNs after mapping (test set):")
print(test_df[['ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN']].isnull().sum())

# If any NaNs exist, fill with the overall train mean as a fallback
overall_train_mean = train_df['ARR_DELAY'].mean()
test_df['ORIGIN_DELAY_MEAN'] = test_df['ORIGIN_DELAY_MEAN'].fillna(overall_train_mean)
test_df['DEST_DELAY_MEAN'] = test_df['DEST_DELAY_MEAN'].fillna(overall_train_mean)
test_df['CARRIER_DELAY_MEAN'] = test_df['CARRIER_DELAY_MEAN'].fillna(overall_train_mean)

NaNs after mapping (test set):
ORIGIN_DELAY_MEAN     1088
DEST_DELAY_MEAN       1102
CARRIER_DELAY_MEAN       0
dtype: int64


In [24]:
test_df['ORIGIN_DELAY_MEAN'] = test_df['ORIGIN_DELAY_MEAN'].fillna(overall_train_mean)
test_df['DEST_DELAY_MEAN'] = test_df['DEST_DELAY_MEAN'].fillna(overall_train_mean)

In [25]:
print("NaNs after fillna:")
print(test_df[['ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN']].isnull().sum())

NaNs after fillna:
ORIGIN_DELAY_MEAN     0
DEST_DELAY_MEAN       0
CARRIER_DELAY_MEAN    0
dtype: int64


In [26]:
import meteostat as ms
from datetime import datetime

start = datetime(2025, 1, 1)
end = datetime(2025, 6, 30, 23, 59)

# Test the nearby-station lookup approach for one airport first
point = ms.Point(35.038932, -106.608262)  # ABQ coordinates
nearby = ms.stations.nearby(point, limit=1)
print(nearby)

                                     name country region  latitude  longitude  \
id                                                                              
72365  Albuquerque International  Airport      US     NM   35.0333     -106.6   

       elevation        timezone  distance  
id                                          
72365       1631  America/Denver     978.8  


In [27]:
# Retry the 4 failed airports with a larger station search (limit=3, take first that has data)
retry_failed = []

for code in failed:
    row = airport_coords_df[airport_coords_df['AIRPORT'] == code].iloc[0]
    point = ms.Point(row['LAT'], row['LON'])
    nearby = ms.stations.nearby(point, limit=5)  # widen the net

    found = False
    for station_id in nearby.index:
        ts = ms.hourly(ms.Station(id=station_id), start, end)
        data = ts.fetch()
        if data is not None and not data.empty:
            weather_data[code] = data
            airport_to_station[code] = station_id
            found = True
            print(f"{code}: matched to backup station {station_id}")
            break

    if not found:
        retry_failed.append(code)

print(f"\nStill failed after retry: {retry_failed}")

HNL: matched to backup station 91182
FSM: matched to backup station 72344
SGU: matched to backup station KSGU0

Still failed after retry: ['DDC']


In [28]:
ddc_flights = combined[(combined['ORIGIN'] == 'DDC') | (combined['DEST'] == 'DDC')]
print(
    f"Flights touching DDC: {len(ddc_flights)} out of {len(combined)} total ({len(ddc_flights) / len(combined) * 100:.4f}%)")

Flights touching DDC: 594 out of 3385822 total (0.0175%)


In [29]:
# since this number is negligible, drop it
print("Before dropping DDC flights:", combined.shape)
combined = combined[(combined['ORIGIN'] != 'DDC') & (combined['DEST'] != 'DDC')].copy()
print("After dropping DDC flights:", combined.shape)


Before dropping DDC flights: (3385822, 33)
After dropping DDC flights: (3385228, 33)


> **TIME TO MERGE ACTUAL WEATHER TO FLIGHTS**

In [30]:
# Carve validation out of the training period itself, keeping chronological order
train_val_split_date = pd.Timestamp('2025-04-01')  # last month of "train" becomes validation

val_df = train_df[train_df['FL_DATE'] >= train_val_split_date].copy()
train_df_final = train_df[train_df['FL_DATE'] < train_val_split_date].copy()

print("Final train:", train_df_final.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Final train: (1611046, 49)
Validation: (577730, 49)
Test: (1197046, 49)


In [31]:
# Recompute target/mean encoding using ONLY the final training set (excluding validation period)
origin_means = train_df_final.groupby('ORIGIN')['ARR_DELAY'].mean()
dest_means = train_df_final.groupby('DEST')['ARR_DELAY'].mean()
carrier_means = train_df_final.groupby('OP_UNIQUE_CARRIER')['ARR_DELAY'].mean()

overall_train_mean = train_df_final['ARR_DELAY'].mean()

for df in [train_df_final, val_df, test_df]:
    df['ORIGIN_DELAY_MEAN'] = df['ORIGIN'].map(origin_means).fillna(overall_train_mean)
    df['DEST_DELAY_MEAN'] = df['DEST'].map(dest_means).fillna(overall_train_mean)
    df['CARRIER_DELAY_MEAN'] = df['OP_UNIQUE_CARRIER'].map(carrier_means).fillna(overall_train_mean)

# Confirm no NaNs anywhere
print("Train NaNs:",
      train_df_final[['ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN']].isnull().sum().sum())
print("Val NaNs:", val_df[['ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN']].isnull().sum().sum())
print("Test NaNs:", test_df[['ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN']].isnull().sum().sum())


Train NaNs: 0
Val NaNs: 0
Test NaNs: 0


In [32]:
feature_cols = [
    # Calendar/seasonality
    'MONTH', 'DAY_OF_WEEK', 'DAY_OF_MONTH', 'IS_WEEKEND', 'IS_HOLIDAY',
    # Scheduled time
    'CRS_DEP_TIME_HOUR', 'CRS_ARR_TIME_HOUR',
    # Route
    'DISTANCE',
    # Frequency encoding
    'ORIGIN_FREQ', 'DEST_FREQ', 'OP_UNIQUE_CARRIER_FREQ',
    # Target/mean encoding
    'ORIGIN_DELAY_MEAN', 'DEST_DELAY_MEAN', 'CARRIER_DELAY_MEAN',
    # Weather
    'temp', 'rhum', 'prcp', 'wspd', 'pres', 'coco'
]

# Classification target
y_train_cls = train_df_final['DELAY_CLASS']
y_val_cls = val_df['DELAY_CLASS']
y_test_cls = test_df['DELAY_CLASS']

# Regression target
y_train_reg = train_df_final['ARR_DELAY_CAPPED']
y_val_reg = val_df['ARR_DELAY_CAPPED']
y_test_reg = test_df['ARR_DELAY_CAPPED']

# Features (same X for both tasks)
X_train = train_df_final[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("\nFeature dtypes:")
print(X_train.dtypes)
print("\nAny NaNs in X_train?", X_train.isnull().sum().sum())

X_train: (1611046, 20)
X_val: (577730, 20)
X_test: (1197046, 20)

Feature dtypes:
MONTH                       int32
DAY_OF_WEEK                 int32
DAY_OF_MONTH                int32
IS_WEEKEND                  int64
IS_HOLIDAY                  int64
CRS_DEP_TIME_HOUR           int64
CRS_ARR_TIME_HOUR           int64
DISTANCE                  float64
ORIGIN_FREQ               float64
DEST_FREQ                 float64
OP_UNIQUE_CARRIER_FREQ    float64
ORIGIN_DELAY_MEAN         float64
DEST_DELAY_MEAN           float64
CARRIER_DELAY_MEAN        float64
temp                      Float64
rhum                      Float64
prcp                      Float64
wspd                      Float64
pres                      Float64
coco                        Int16
dtype: object

Any NaNs in X_train? 0


In [33]:
print("NaN count per column in X_train:")
print(X_train.isnull().sum().sort_values(ascending=False))

NaN count per column in X_train:
MONTH                     0
DAY_OF_WEEK               0
DAY_OF_MONTH              0
IS_WEEKEND                0
IS_HOLIDAY                0
CRS_DEP_TIME_HOUR         0
CRS_ARR_TIME_HOUR         0
DISTANCE                  0
ORIGIN_FREQ               0
DEST_FREQ                 0
OP_UNIQUE_CARRIER_FREQ    0
ORIGIN_DELAY_MEAN         0
DEST_DELAY_MEAN           0
CARRIER_DELAY_MEAN        0
temp                      0
rhum                      0
prcp                      0
wspd                      0
pres                      0
coco                      0
dtype: int64


In [34]:
# BASELINES MODELS

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Scale features — logistic/linear regression are sensitive to feature scale, tree models later won't need this
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# --- Baseline classifier ---
clf_baseline = LogisticRegression(max_iter=1000)
clf_baseline.fit(X_train_scaled, y_train_cls)

train_acc = clf_baseline.score(X_train_scaled, y_train_cls)
val_acc = clf_baseline.score(X_val_scaled, y_val_cls)
print(f"Logistic Regression — Train acc: {train_acc:.4f}, Val acc: {val_acc:.4f}")

# --- Baseline regressor ---
reg_baseline = LinearRegression()
reg_baseline.fit(X_train_scaled, y_train_reg)

val_pred = reg_baseline.predict(X_val_scaled)
mae = mean_absolute_error(y_val_reg, val_pred)
rmse = np.sqrt(mean_squared_error(y_val_reg, val_pred))
r2 = r2_score(y_val_reg, val_pred)

print(f"\nLinear Regression — Val MAE: {mae:.2f}")
print(f"Linear Regression — Val RMSE: {rmse:.2f}")
print(f"Linear Regression — Val R²: {r2:.4f}")

Logistic Regression — Train acc: 0.4930, Val acc: 0.5338

Linear Regression — Val MAE: 24.05
Linear Regression — Val RMSE: 40.88
Linear Regression — Val R²: 0.0317


In [35]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, classification_report

# --- LightGBM classifier ---
# Note: no need to scale features for tree-based models
clf_lgb = lgb.LGBMClassifier(n_estimators=200, random_state=42)
clf_lgb.fit(X_train, y_train_cls)

val_pred_cls = clf_lgb.predict(X_val)
print("LightGBM Classifier:")
print(f"Val Accuracy: {accuracy_score(y_val_cls, val_pred_cls):.4f}")
print(f"Val Macro-F1: {f1_score(y_val_cls, val_pred_cls, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(y_val_cls, val_pred_cls))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061282 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1920
[LightGBM] [Info] Number of data points in the train set: 1611046, number of used features: 20
[LightGBM] [Info] Start training from score -1.657384
[LightGBM] [Info] Start training from score -1.134648
[LightGBM] [Info] Start training from score -0.717793
LightGBM Classifier:
Val Accuracy: 0.5320
Val Macro-F1: 0.3407

Classification Report:
              precision    recall  f1-score   support

     Delayed       0.37      0.06      0.11    109771
       Early       0.43      0.16      0.24    159485
     On-time       0.55      0.89      0.68    308474

    accuracy                           0.53    577730
   macro avg       0.45      0.37      0.34    577730
weighted avg       0.48      0.53      0.45    577730



In [36]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_val_cls, val_pred_cls, labels=['Early', 'On-time', 'Delayed'])
cm_df = pd.DataFrame(cm, index=['Actual: Early', 'Actual: On-time', 'Actual: Delayed'],
                     columns=['Pred: Early', 'Pred: On-time', 'Pred: Delayed'])
print(cm_df)

# Feature importance — which features is the model actually relying on?
importance = pd.Series(clf_lgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nFeature importance:")
print(importance)

                 Pred: Early  Pred: On-time  Pred: Delayed
Actual: Early          26025         129593           3867
Actual: On-time        26371         274585           7518
Actual: Delayed         8525          94486           6760

Feature importance:
DAY_OF_MONTH              2576
DISTANCE                  1346
ORIGIN_FREQ               1311
DEST_FREQ                 1261
ORIGIN_DELAY_MEAN         1258
DEST_DELAY_MEAN           1240
MONTH                     1212
temp                      1183
pres                      1155
DAY_OF_WEEK                859
CRS_DEP_TIME_HOUR          811
OP_UNIQUE_CARRIER_FREQ     760
rhum                       650
CRS_ARR_TIME_HOUR          619
CARRIER_DELAY_MEAN         618
coco                       560
wspd                       371
IS_HOLIDAY                 130
prcp                        80
IS_WEEKEND                   0
dtype: int32


In [37]:
combined_binary_train = y_train_cls.apply(lambda x: 'Delayed' if x == 'Delayed' else 'Not Delayed')
combined_binary_val = y_val_cls.apply(lambda x: 'Delayed' if x == 'Delayed' else 'Not Delayed')

clf_lgb_binary = lgb.LGBMClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    class_weight='balanced', random_state=42
)
clf_lgb_binary.fit(X_train, combined_binary_train)

pred_binary = clf_lgb_binary.predict(X_val)
print(f"Binary Accuracy: {accuracy_score(combined_binary_val, pred_binary):.4f}")
print(f"Binary Macro-F1: {f1_score(combined_binary_val, pred_binary, average='macro'):.4f}")
print(classification_report(combined_binary_val, pred_binary))

[LightGBM] [Info] Number of positive: 1303921, number of negative: 307125
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057121 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1920
[LightGBM] [Info] Number of data points in the train set: 1611046, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Binary Accuracy: 0.6667
Binary Macro-F1: 0.5683
              precision    recall  f1-score   support

     Delayed       0.28      0.50      0.36    109771
 Not Delayed       0.86      0.71      0.77    467959

    accuracy                           0.67    577730
   macro avg       0.57      0.60      0.57    577730
weighted avg       0.75      0.67      0.70    577730



In [38]:
# pip install xgboost --break-system-packages   (if not already installed)
import xgboost as xgb

# --- XGBoost binary classifier ---
# XGBoost needs numeric labels, not strings — encode 'Delayed'/'Not Delayed' as 1/0
y_train_binary_num = (combined_binary_train == 'Delayed').astype(int)
y_val_binary_num = (combined_binary_val == 'Delayed').astype(int)

clf_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    scale_pos_weight=(y_train_binary_num == 0).sum() / (y_train_binary_num == 1).sum(),
    # XGBoost's equivalent of class_weight='balanced' for binary
    random_state=42,
    eval_metric='logloss'
)
clf_xgb.fit(X_train, y_train_binary_num)

val_pred_xgb = clf_xgb.predict(X_val)
print("XGBoost Binary Classifier:")
print(f"Val Accuracy: {accuracy_score(y_val_binary_num, val_pred_xgb):.4f}")
print(f"Val Macro-F1: {f1_score(y_val_binary_num, val_pred_xgb, average='macro'):.4f}")
print(classification_report(y_val_binary_num, val_pred_xgb, target_names=['Not Delayed', 'Delayed']))

XGBoost Binary Classifier:
Val Accuracy: 0.6687
Val Macro-F1: 0.5681
              precision    recall  f1-score   support

 Not Delayed       0.86      0.71      0.78    467959
     Delayed       0.28      0.49      0.36    109771

    accuracy                           0.67    577730
   macro avg       0.57      0.60      0.57    577730
weighted avg       0.75      0.67      0.70    577730



In [39]:
reg_xgb = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)
reg_xgb.fit(X_train, y_train_reg)

val_pred_reg_xgb = reg_xgb.predict(X_val)
mae_xgb = mean_absolute_error(y_val_reg, val_pred_reg_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_val_reg, val_pred_reg_xgb))
r2_xgb = r2_score(y_val_reg, val_pred_reg_xgb)

print(f"\nXGBoost Regressor — Val MAE: {mae_xgb:.2f}")
print(f"XGBoost Regressor — Val RMSE: {rmse_xgb:.2f}")
print(f"XGBoost Regressor — Val R²: {r2_xgb:.4f}")


XGBoost Regressor — Val MAE: 23.56
XGBoost Regressor — Val RMSE: 41.21
XGBoost Regressor — Val R²: 0.0163


In [40]:
reg_lgb = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)
reg_lgb.fit(X_train, y_train_reg)

val_pred_reg_lgb = reg_lgb.predict(X_val)
mae_lgb = mean_absolute_error(y_val_reg, val_pred_reg_lgb)
rmse_lgb = np.sqrt(mean_squared_error(y_val_reg, val_pred_reg_lgb))
r2_lgb = r2_score(y_val_reg, val_pred_reg_lgb)

print(f"LightGBM Regressor — Val MAE: {mae_lgb:.2f}")
print(f"LightGBM Regressor — Val RMSE: {rmse_lgb:.2f}")
print(f"LightGBM Regressor — Val R²: {r2_lgb:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.145264 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1920
[LightGBM] [Info] Number of data points in the train set: 1611046, number of used features: 20
[LightGBM] [Info] Start training from score 3.816396
LightGBM Regressor — Val MAE: 23.21
LightGBM Regressor — Val RMSE: 41.04
LightGBM Regressor — Val R²: 0.0244


In [41]:
from sklearn.metrics import f1_score

val_pred_logreg = clf_baseline.predict(X_val_scaled)
logreg_macro_f1 = f1_score(y_val_cls, val_pred_logreg, average='macro')
print(f"Logistic Regression — Val Macro-F1: {logreg_macro_f1:.4f}")

Logistic Regression — Val Macro-F1: 0.2780


In [42]:
import pandas as pd

# --- Classification results table ---
classification_results = pd.DataFrame([
    {
        'Model': 'Logistic Regression (baseline)',
        'Task': '3-class',
        'Accuracy': 0.5338,
        'Macro-F1': logreg_macro_f1
    },
    {
        'Model': 'LightGBM (unweighted)',
        'Task': '3-class',
        'Accuracy': 0.5320,
        'Macro-F1': 0.3407,
    },
    {
        'Model': 'LightGBM',
        'Task': 'Binary (Delayed vs Not)',
        'Accuracy': 0.6667,
        'Macro-F1': 0.5683,
    },
    {
        'Model': 'XGBoost',
        'Task': 'Binary (Delayed vs Not)',
        'Accuracy': 0.6687,
        'Macro-F1': 0.5681,
    },
])

print("=== Classification Benchmarking ===")
print(classification_results.to_string(index=False))

# --- Regression results table ---
regression_results = pd.DataFrame([
    {
        'Model': 'Linear Regression (baseline)',
        'MAE': 24.05,
        'RMSE': 40.88,
        'R2': 0.0317,
    },
    {
        'Model': 'XGBoost Regressor',
        'MAE': 23.56,
        'RMSE': 41.21,
        'R2': 0.0163,
    },
    {
        'Model': 'LightGBM Regressor',
        'MAE': mae_lgb,
        'RMSE': rmse_lgb,
        'R2': r2_lgb,
    },
])

print("\n=== Regression Benchmarking ===")
print(regression_results.to_string(index=False))

=== Classification Benchmarking ===
                         Model                    Task  Accuracy  Macro-F1
Logistic Regression (baseline)                 3-class    0.5338  0.277982
         LightGBM (unweighted)                 3-class    0.5320  0.340700
                      LightGBM Binary (Delayed vs Not)    0.6667  0.568300
                       XGBoost Binary (Delayed vs Not)    0.6687  0.568100

=== Regression Benchmarking ===
                       Model       MAE      RMSE       R2
Linear Regression (baseline) 24.050000 40.880000 0.031700
           XGBoost Regressor 23.560000 41.210000 0.016300
          LightGBM Regressor 23.210862 41.035537 0.024358


In [43]:
import joblib
import os

os.makedirs(os.path.join(DATASETS_DIR, "..", "models"), exist_ok=True)
model_dir = os.path.join(DATASETS_DIR, "..", "models")

joblib.dump(clf_lgb, os.path.join(model_dir, "delay_classifier.pkl"))  # your binary LightGBM classifier
joblib.dump(reg_lgb, os.path.join(model_dir, "delay_regressor.pkl"))  # LightGBM regressor
joblib.dump(scaler, os.path.join(model_dir,
                                 "scaler.pkl"))  # only needed if backend also serves the linear baseline; skip if not

# Also save your encoding lookup tables — the API will need these to encode incoming requests the same way
import pickle

encoders = {
    'origin_freq': combined['ORIGIN'].value_counts(normalize=True).to_dict(),
    'dest_freq': combined['DEST'].value_counts(normalize=True).to_dict(),
    'carrier_freq': combined['OP_UNIQUE_CARRIER'].value_counts(normalize=True).to_dict(),
    'origin_delay_mean': origin_means.to_dict(),
    'dest_delay_mean': dest_means.to_dict(),
    'carrier_delay_mean': carrier_means.to_dict(),
    'overall_train_mean': overall_train_mean,
}
with open(os.path.join(model_dir, "encoders.pkl"), 'wb') as f:
    encoders['route_distances'] = {
        f"{o}-{d}": float(dist)
        for (o, d), dist in combined.groupby(['ORIGIN', 'DEST'])['DISTANCE'].first().items()
    }
    pickle.dump(encoders, f)

print("Saved models and encoders to:", model_dir)

Saved models and encoders to: D:/CitrusBits/pythonic-rebirth\datasets\..\models


In [44]:
print("route_distances sample:", list(encoders['route_distances'].items())[:3])

route_distances sample: [('ABE-BNA', 685.0), ('ABE-CLT', 481.0), ('ABE-DEN', 1539.0)]
